# Health Retailers — Demand Forecasting (cahier §11.1)
## Complete, executable Data Preparation Lab

**Objective:** transform raw sales exports, the product catalogue and two external sources (weather, epidemics)
into a traceable, model-ready dataset to **forecast the quantity sold next week (N+1) per product family or brand**.

**Analytical grain:** **one row = one week × one family** (or one brand, see `LEVEL`). The target is the quantity
sold the **following** week.

**Data used:** the real export (30/09/2024 → 30/04/2025, 7 months) **plus** the generated continuation
(01/05/2025 → 29/09/2026) so that the series covers two years and a full seasonal cycle. Real and synthetic weeks are
**tagged by date**, and every conclusion about external variables is measured on **real weeks only**.

**Team division:** Person 1 = sales sources, cleaning, weekly series and lag features. Person 2 = calendar, weather,
epidemics, feature selection, encoding and PCA. Both validate the final dataset.

# 1 — Business objective and forecasting setup

**Problem:** the purchasing manager orders every week. Ordering too little creates stockouts; ordering too much
ties up cash and creates expired products. A reliable estimate of next week's demand per family supports
replenishment.

**Forecast moment:** the forecast is made **at the end of week t** for **week t+1**. Consequently:
- sales of week t and before are known → lags and moving averages are legitimate;
- the calendar of week t+1 (public holidays, month) is known in advance → legitimate;
- the weather of week t is observed; the weather of week t+1 would itself be a forecast → **not used**;
- epidemic indicators are published with ~1 week of delay → only **week t−1** values are used.

**Full workflow**
1. **Data Understanding:** profile every source separately.
2. **Data Cleaning:** types, decimal commas, dates, drafts, duplicates, double-counted invoices, quarantine.
3. **Data Transformation:** controlled joins (line → document → product → family), signed quantities,
   weekly aggregation on a complete calendar.
4. **Feature Engineering:** lags, moving average, price and discount signals, calendar, weather, epidemics.
5. **Feature Selection:** relevance on real weeks, redundancy, missingness, leakage audit.
6. **Dimensionality Reduction:** imputation, encoding, scaling, PCA of the correlated weather block.
7. **Optional forecasting extension:** baseline vs model, with and without external variables.
8. **Validation and reporting.**

## How to study this notebook

For each stage, read the explanation **before** running the code. It tells you **why** the step is needed for
demand forecasting, **what** the important commands do, **what output** to expect and **what to say** to the
professor. Every number and chart below comes from executing the code on the project files.

## 0 — Setup and source files

| Source | Purpose | Key fields |
|---|---|---|
| `Documents.csv` | Document header: date, type, state | `Id`, `DocumentDate`, `DocumentType_Id`, `State` |
| `documentItemm.csv` | Product lines (the sales grain) | `Document_Id`, `ProductStock_Id`, `Quantity`, `PrixTotalHt`, `SumDiscountTotal` |
| `DocumentTypes.csv` | Meaning of each document type | `Id`, `Name`, `StockFactor`, `IsSellType` |
| `RelatedDocuments.csv` | Delivery notes grouped into an invoice | `Document1_Id`, `Document2_Id` |
| `ProductStocks.csv` | Stock line → product | `Id`, `Product_Id` |
| `cleaned/products.csv` | Family and brand per product (key rebuilt by `script_cleaning_Produits.py`) | `Product_Id`, `FamilyGroup`, `Marque` |
| `meteo_paris_open-meteo.json` | Daily weather, Paris (Open-Meteo, CC BY 4.0) | temperature, rain, sunshine, radiation |
| `sentinelles_inc-{3,25,6,7}-PAY.csv` | Weekly epidemics, France (Réseau Sentinelles) | `week`, `inc100` |

**Why the cleaned catalogue?** The raw `Produits.csv` has **no numeric `Product_Id`**: the key was rebuilt during
cleaning (product events, barcodes, exact names). Re-doing that matching here would duplicate
`script_cleaning_Produits.py`, so we reuse its output and say so.

### Guided code explanation — Setup

1. `PROJECT` points to the clone of the repository; `RAW_DIR`, `SYNTH_DIR`, `EXT_DIR`, `CLEAN_DIR` are its folders.
2. `REAL_END` is the last day of the real export: everything after it is synthetic.
3. `INCLUDE_SYNTHETIC = True` because a weekly forecast needs more than 30 weeks of history.
4. `OUTPUT_DIR` receives every generated file; the raw CSVs are never overwritten.

**What you should see:** four folders reported as `found`.

In [1]:
# STEP 0: Import the libraries and define every input/output path in one place.
# Raw exports are read from the project folders and are never modified.
# All generated evidence (profiles, audits, matrices, figures) goes to outputs/demand_forecast/.
# Change PROJECT if the health-retailers repository is stored elsewhere on your computer.

# Run once if needed:
# %pip install pandas numpy matplotlib scikit-learn

from pathlib import Path
import io, json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# Locate the data branch checked out beside this branch, or set HEALTH_RETAILERS_DATA.
import os
_candidates = [Path(os.environ['HEALTH_RETAILERS_DATA']).expanduser()] if os.environ.get('HEALTH_RETAILERS_DATA') else []
for _base in (Path.cwd(), *Path.cwd().parents):
    _candidates.extend([_base / 'data', _base.parent / 'data'])
PROJECT = next((_p.resolve() for _p in _candidates if (_p / 'original').is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError('Check out the data branch beside ml-notebooks, or set HEALTH_RETAILERS_DATA to its root.')
RAW_DIR = PROJECT / 'original'                       # branch data-original: the 29 raw CSV files
SYNTH_DIR = PROJECT / 'generated'   # branch data-external: generated months
EXT_DIR = PROJECT / 'external'                        # weather and epidemics downloads
CLEAN_DIR = PROJECT / 'cleaned'                      # branch data-cleaned (product catalogue key)
OUTPUT_DIR = Path('outputs') / 'demand_forecast'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REAL_START = pd.Timestamp('2024-09-30')
REAL_END = pd.Timestamp('2025-04-30 23:59:59')       # last day of the real export
SYNTH_END = pd.Timestamp('2026-09-29 23:59:59')      # last day of the generated data
INCLUDE_SYNTHETIC = True                  # True = add the generated months (tagged, never mixed silently)

for folder in [RAW_DIR, SYNTH_DIR, EXT_DIR, CLEAN_DIR]:
    print(f'{str(folder):60s}', 'found' if folder.exists() else 'MISSING')

data\original found
data\generated found
data\external found
data\cleaned found


### Guided code explanation — Reading and conversion helpers

**Why this step exists:** the 29 raw files were not exported consistently. Some are UTF-8, others Windows-1252
or DOS code page 850; some use `;`, others `,`; amounts use a decimal comma; dates mix `MM/DD/YYYY` and
`DD/MM/YYYY`. One set of helpers guarantees that every file follows the same rules.

- `read_text()` tries UTF-8 first, then keeps the legacy code page that produces real French letters
  (`pièce` instead of `pi�ce`).
- `load_csv()` detects `;` or `,` from the header line and reads **every column as text** (`dtype='string'`),
  so identifiers and amounts are never converted by accident before we inspect them.
- `parse_amount()` turns `42,38` into `42.38`; unreadable values become `NaN` and are counted.
- `parse_bool()` maps `VRAI/FAUX`, `TRUE/FALSE`, `1/0` to one Boolean type.
- `parse_date()` uses an **explicit format** per column. We never let pandas guess between day-first and
  month-first, because `03/04/2025` is ambiguous.
- `profile()` returns the Data Understanding evidence: type, missing values, distinct values.

**What to tell your professor:** "Every conversion is explicit and audited: a value that cannot be parsed
becomes missing and is counted, it is never invented."

In [2]:
# Helpers shared by every step of the notebook.
# Every column is read as text first; conversions are explicit and audited later.

def read_text(path):
    raw = path.read_bytes()
    try:
        return raw.decode('utf-8-sig')
    except UnicodeDecodeError:
        # Legacy export: keep the code page that yields real French letters.
        candidates = [raw.decode(cp, errors='replace') for cp in ('cp1252', 'cp850')]
        score = lambda t: sum(t.count(c) for c in 'éèàçêù°') - 3 * sum(t.count(c) for c in 'Š‚ø')
        return max(candidates, key=score)

def load_csv(path):
    text = read_text(path)
    header = text.split('\n', 1)[0]
    sep = ';' if header.count(';') > header.count(',') else ','
    df = pd.read_csv(io.StringIO(text), sep=sep, dtype='string')
    df.columns = [str(c).strip() for c in df.columns]
    return df

def parse_amount(series):
    s = series.astype('string').str.strip().str.replace('\u00a0', '', regex=False).str.replace(' ', '', regex=False)
    return pd.to_numeric(s.str.replace(',', '.', regex=False), errors='coerce').astype('float64')

def parse_bool(series):
    mapping = {'vrai': True, 'true': True, '1': True, 'faux': False, 'false': False, '0': False}
    return series.astype('string').str.strip().str.lower().map(mapping).astype('boolean')

def parse_date(series, fmt):
    return pd.to_datetime(series.astype('string').str.strip(), format=fmt, errors='coerce')

def profile(df):
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing_count': df.isna().sum(),
        'missing_%': (100 * df.isna().mean()).round(2),
        'unique_values': df.nunique(dropna=True),
    })

def safe_left_merge(left, right, left_on, right_on, label):
    """Left join that refuses to multiply rows: the lookup key must be unique."""
    right = right.dropna(subset=[right_on])
    if right[right_on].duplicated().any():
        raise ValueError(f'{label}: lookup key {right_on} is not unique, aggregate it first')
    n = len(left)
    out = left.merge(right, how='left', left_on=left_on, right_on=right_on,
                     suffixes=('', '_' + label), validate='many_to_one')
    assert len(out) == n, f'{label} multiplied rows!'
    matched = out[right_on].notna().mean() * 100
    print(f'{label:28s} {n:>7,} rows kept | {matched:5.1f} % matched')
    return out

conversion_log = []
def convert(df, name, amounts=(), bools=(), dates=None):
    """Apply the conversions and record how many non-empty values could not be parsed."""
    df = df.copy()
    for col in amounts:
        if col in df:
            before = df[col].notna().sum(); df[col] = parse_amount(df[col])
            conversion_log.append([name, col, 'amount', int(before - df[col].notna().sum())])
    for col in bools:
        if col in df:
            before = df[col].notna().sum(); df[col] = parse_bool(df[col])
            conversion_log.append([name, col, 'boolean', int(before - df[col].notna().sum())])
    for col, fmt in (dates or {}).items():
        if col in df:
            before = df[col].notna().sum(); df[col] = parse_date(df[col], fmt)
            conversion_log.append([name, col, f'date {fmt}', int(before - df[col].notna().sum())])
    return df

print('Helpers ready.')

Helpers ready.


## 1 — Data Understanding: profile every dataset separately

We first produce **row counts, columns, types, missing values, distinct values and duplicates** for each source,
before any join.

### Guided code explanation — Profiling the sources

**Business question:** which sources describe *what was sold, when, and which product*, and which quality problems
must be solved before building a weekly series?

- `raw[...]` holds the real export; `synth[...]` the generated months (same file names and formats).
- `profile(df)` gives type, missing count and percentage, and distinct values per column.
- `df.duplicated().sum()` counts **exactly identical rows**.
- The weather JSON is turned into a table with one row per day; each Sentinelles file has comment lines starting
  with `#`, which are skipped.

**How to interpret:** `documentItemm.csv` has many 100 % empty columns (they will simply not be used);
`DocumentDate` has no time (always `0:00`); amounts are text with a decimal comma.

**Presentation use:** show the profile of `documentItemm.csv` and one line per supporting source.

In [3]:
# STEP 1: Read each dataset independently before attempting any joins.
SOURCES = {'documents': 'Documents.csv', 'items': 'documentItemm.csv', 'doc_types': 'DocumentTypes.csv',
           'related': 'RelatedDocuments.csv', 'product_stocks': 'ProductStocks.csv'}
raw = {k: load_csv(RAW_DIR / f) for k, f in SOURCES.items()}
synth = {k: load_csv(SYNTH_DIR / f) for k, f in SOURCES.items()
         if INCLUDE_SYNTHETIC and (SYNTH_DIR / f).exists()}
catalogue = pd.read_csv(CLEAN_DIR / 'products.csv', dtype='string')

overview = []
for label, tables in (('real', raw), ('synthetic', synth)):
    for key, df in tables.items():
        overview.append([SOURCES[key], label, len(df), df.shape[1], int(df.duplicated().sum()),
                         int((df.isna().mean() == 1).sum())])
overview.append(['cleaned/products.csv', 'reference', len(catalogue), catalogue.shape[1],
                 int(catalogue.duplicated().sum()), int((catalogue.isna().mean() == 1).sum())])
overview = pd.DataFrame(overview, columns=['file', 'origin', 'rows', 'columns', 'exact_duplicates', 'empty_columns'])
display(overview)
overview.to_csv(OUTPUT_DIR / 'sources_overview.csv', index=False)

for key in ['items', 'documents']:
    print(f'\n=== {SOURCES[key]} (real): {len(raw[key]):,} rows × {raw[key].shape[1]} columns ===')
    display(raw[key].head(3))
    p = profile(raw[key]); display(p)
    p.to_csv(OUTPUT_DIR / f'profile_{key}.csv', index_label='column')
display(raw['doc_types'][['Id', 'Name', 'StockFactor', 'IsSellType']])

,file,origin,rows,columns,exact_duplicates,empty_columns
0,Documents.csv,real,12542,35,0,2
1,documentItemm.csv,real,38705,34,0,10
2,DocumentTypes.csv,real,15,16,0,0
3,RelatedDocuments.csv,real,864,4,0,0
4,ProductStocks.csv,real,6842,5,0,0
5,Documents.csv,synthetic,32874,35,0,4
6,documentItemm.csv,synthetic,82665,34,0,10
7,ProductStocks.csv,synthetic,6842,5,0,0
8,cleaned/products.csv,reference,7792,20,0,0



=== documentItemm.csv (real): 38,705 rows × 34 columns ===


,Id,DisplayOrder,Description,Quantity,PrixHt,Tva,PrixTotalHt,PrixTotalTtc,DiscountPourcentage,DiscountAmount,Document_Id,ProductStock_Id,ProductPackage_Id,TotalTva,IncludedTaxes,TotalTaxes,SumPrixNetHt,SumQuantityUnit,SumDiscountTotal,SumPrixTotalHt,SumTotalTva,SumIncludedTaxes,SumTotalTaxes,SumPrixTotalTtc,PrixTtc,ForcedPrixTtc,SumPrixNetTtc,Extra,PrixTvc,ProductStockLocation_Id,BenTaxCategory_Id,NerTaxCategory_Id,TtcDiscountPercentage,TtcDiscountAmount
0,1,2,"<p style=""margin-top: 0px;margin-bottom: 0px;l...",1,"34,25",19,"27,4","32,61",20,<NA>,1,4157,<NA>,"5,21",<NA>,<NA>,"27,4",1,"6,85","27,4","5,206",0,0,"32,606","32,61",<NA>,"32,606",{},<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2,2,"<p style=""margin-top: 0px;margin-bottom: 0px;l...",1,"44,51",19,"35,61","42,38",20,<NA>,2,387,<NA>,"6,77",<NA>,<NA>,"35,608",1,"8,9","35,61","6,7659",0,0,"42,3759","42,38",<NA>,"42,3759",{},<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,3,2,"<p style=""margin-top: 0px;margin-bottom: 0px;l...",1,"12,3",19,"9,84","11,71",20,<NA>,3,3324,<NA>,"1,87",<NA>,<NA>,"9,84",1,"2,46","9,84","1,8696",0,0,"11,7096","11,71",<NA>,"11,7096",{},<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


,dtype,missing_count,missing_%,unique_values
Id,string,0,0.00,38705
DisplayOrder,string,0,0.00,911
Description,string,0,0.00,5573
Quantity,string,1035,2.67,61
PrixHt,string,1079,2.79,6385
Tva,string,2472,6.39,2
PrixTotalHt,string,1079,2.79,11416
PrixTotalTtc,string,1079,2.79,11651
DiscountPourcentage,string,15077,38.95,88
DiscountAmount,string,38705,100.00,0



=== Documents.csv (real): 12,542 rows × 35 columns ===


,Id,DocumentNumber,DocumentDate,State,TotalHt,TotalNetHt,DiscountPercentage,TotalTva,TotalTtc,Timbre,FinalPrice,IsDelivered,Observation,DocumentType_Id,Transaction_Id,PaymentMode_Id,ProductPriceName_Id,DefaultStock_Id,UserComment,Attachments_Id,DiscountAmount,TotalDiscount,IncludedTaxes,TotalTaxes,TotalExtraTaxes,CreatedDate,CreatedBy,ModifiedDate,ModifiedBy,ValidatedDate,ValidatedBy,Extra,TotalRetenueSurHt,TotalRetenueSurTva,TotalRetenueSurTtc
0,1,VC0001/24,9/30/2024 0:00,1,27.4,27.4,<NA>,5.21,32.61,0,32.61,FALSE,<NA>,6,1,11,1,1,<NA>,<NA>,<NA>,0,0,0,0,9/30/2024 8:28,3,9/30/2024 8:28,3,9/30/2024 8:28,3,"{""cosld"":0.0}",0,0,0
1,2,VC0002/24,9/30/2024 0:00,1,35.61,35.61,<NA>,6.77,42.38,0,42.38,FALSE,<NA>,6,2,5,1,1,<NA>,<NA>,<NA>,0,0,0,0,9/30/2024 7:51,3,9/30/2024 7:51,3,9/30/2024 7:51,3,"{""cosld"":32.6100}",0,0,0
2,3,VC0003/24,9/30/2024 0:00,1,9.84,9.84,<NA>,1.87,11.71,0,11.71,FALSE,<NA>,6,3,11,1,1,<NA>,<NA>,<NA>,0,0,0,0,9/30/2024 7:53,3,9/30/2024 7:53,3,9/30/2024 7:53,3,"{""cosld"":0.0}",0,0,0


,dtype,missing_count,missing_%,unique_values
Id,string,0,0.00,12542
DocumentNumber,string,0,0.00,12542
DocumentDate,string,0,0.00,176
State,string,0,0.00,2
TotalHt,string,0,0.00,9086
TotalNetHt,string,0,0.00,9082
DiscountPercentage,string,12438,99.17,14
TotalTva,string,0,0.00,8000
TotalTtc,string,0,0.00,9170
Timbre,string,0,0.00,2


,Id,Name,StockFactor,IsSellType
0,1,FactureClient,-1,VRAI
1,2,Proforma,0,VRAI
2,3,CommandeClient,0,VRAI
3,4,AvoirClient,1,VRAI
4,5,BonLivraison,-1,VRAI
5,6,VenteComptoir,-1,VRAI
6,7,BonRetour,1,VRAI
7,8,FactureDeRoute,0,VRAI
8,20,FactureFour,1,FAUX
9,21,CommandFour,0,FAUX


In [4]:
# STEP 1B: External sources. Weather = one row per day; epidemics = one row per ISO week and indicator.
daily = json.loads((EXT_DIR / 'meteo_paris_open-meteo.json').read_text(encoding='utf-8'))['daily']
weather = pd.DataFrame(daily).rename(columns={
    'time': 'date', 'temperature_2m_mean': 'temp_mean', 'temperature_2m_min': 'temp_min',
    'temperature_2m_max': 'temp_max', 'precipitation_sum': 'rain_mm',
    'sunshine_duration': 'sunshine_s', 'shortwave_radiation_sum': 'radiation_mj'})
weather['date'] = pd.to_datetime(weather['date'])
print(f'Weather: {len(weather)} days from {weather.date.min():%d/%m/%Y} to {weather.date.max():%d/%m/%Y}')
display(profile(weather))

SENTINELLES = {'3': 'flu', '25': 'resp_infection', '6': 'diarrhoea', '7': 'chickenpox'}
epi_parts = []
for code_, name in SENTINELLES.items():
    df = pd.read_csv(EXT_DIR / f'sentinelles_inc-{code_}-PAY.csv', comment='#', dtype='string')
    df['indicator'] = name
    epi_parts.append(df[['week', 'indicator', 'inc100']])
epidemics_raw = pd.concat(epi_parts, ignore_index=True)
print(f'Epidemics: {len(epidemics_raw):,} rows (weeks × indicators), weeks {epidemics_raw.week.min()} → {epidemics_raw.week.max()}')
display(epidemics_raw.head())

Weather: 759 days from 01/09/2024 to 29/09/2026


,dtype,missing_count,missing_%,unique_values
date,datetime64[us],0,0.0,759
temp_mean,float64,0,0.0,265
temp_min,float64,0,0.0,242
temp_max,float64,0,0.0,297
rain_mm,float64,0,0.0,118
sunshine_s,float64,0,0.0,647
radiation_mj,float64,0,0.0,658


Epidemics: 6,294 rows (weeks × indicators), weeks 198444 → 202646


,week,indicator,inc100
0,202646,flu,84
1,202645,flu,79
2,202644,flu,80
3,202643,flu,84
4,202642,flu,85


## 2 — Data Cleaning: standardize, flag, never silently correct

**Person 1:** documents and lines. Rules:
- Decimal comma → decimal point for quantities, prices, totals and discounts.
- `DocumentDate` and `CreatedDate` are **month/day/year** (`9/30/2024 8:28`): parsed with an explicit format.
- `State = 1` means validated, `0` means draft: drafts are **not sales**.
- Only exact duplicate rows may be dropped automatically.
- Real and synthetic rows are concatenated **with an origin tag** derived from the date.

### Guided code explanation — Standardizing formats

- `convert()` applies `parse_amount`, `parse_bool`, `parse_date` and records in `conversion_log` how many
  non-empty values could not be parsed (they would become `NaN`).
- `pd.concat([real, synthetic])` stacks the two periods; `assert ...is_unique` proves the generated document ids
  do not collide with the real ones.
- `origin` is computed **from the date only** (`≤ 30/04/2025` → real). There is no hidden source column.

**What to inspect:** the conversion audit must show **0** unparseable values; otherwise a format assumption is wrong.

**Presentation sentence:** "We converted every amount and date with an explicit rule and audited conversion failures."

In [5]:
# STEP 2: Convert types, stack real + synthetic, tag the origin by date.
def stack(key):
    parts = [raw[key]] + ([synth[key]] if key in synth else [])
    return pd.concat(parts, ignore_index=True)

docs = convert(stack('documents'), 'documents', amounts=['TotalHt', 'TotalNetHt'],
               dates={'DocumentDate': '%m/%d/%Y %H:%M', 'CreatedDate': '%m/%d/%Y %H:%M'})
items = convert(stack('items'), 'items', amounts=['Quantity', 'PrixHt', 'PrixTotalHt', 'DiscountPourcentage',
                                                 'SumDiscountTotal'])
doc_types = convert(raw['doc_types'], 'doc_types', amounts=['StockFactor'], bools=['IsSellType'])

for name, df in (('documents', docs), ('items', items)):
    dup = int(df.duplicated().sum())
    print(f'{name}: {len(df):,} rows, exact duplicates = {dup}')
docs = docs.drop_duplicates(); items = items.drop_duplicates()
assert docs['Id'].is_unique and items['Id'].is_unique, 'real and synthetic ids collide'

docs['origin'] = np.where(docs['DocumentDate'] <= REAL_END, 'real', 'synthetic')
display(docs['origin'].value_counts().to_frame('documents'))
conversion_audit = pd.DataFrame(conversion_log, columns=['dataset', 'column', 'conversion', 'unparseable_non_null'])
display(conversion_audit)
conversion_audit.to_csv(OUTPUT_DIR / 'conversion_audit.csv', index=False)

documents: 45,416 rows, exact duplicates = 0


items: 121,370 rows, exact duplicates = 0


,documents
origin,
synthetic,32874
real,12542


,dataset,column,conversion,unparseable_non_null
0,documents,TotalHt,amount,0
1,documents,TotalNetHt,amount,0
2,documents,DocumentDate,date %m/%d/%Y %H:%M,0
3,documents,CreatedDate,date %m/%d/%Y %H:%M,0
4,items,Quantity,amount,0
5,items,PrixHt,amount,0
6,items,PrixTotalHt,amount,0
7,items,DiscountPourcentage,amount,0
8,items,SumDiscountTotal,amount,0
9,doc_types,StockFactor,amount,0


### Guided code explanation — Which documents are sales? Quality flags and quarantine

`DocumentTypes.csv` explains the codes. For demand we keep:
- **sales:** `6` counter sale (VenteComptoir), `5` delivery note (BonLivraison), `1` customer invoice (FactureClient);
- **returns:** `4` customer credit note (AvoirClient), `7` return note (BonRetour) → **negative** quantities;
- everything else (purchases, quotes, orders, stock moves) is not customer demand.

Flags on each line (they are **counted, not hidden**):
- `flag_no_document`: the line points to a document that does not exist.
- `flag_comment_line`: no product and no amount — a layout/comment line of an invoice.
- `flag_draft`: the document is not validated (`State = 0`).
- `flag_grouped_invoice`: a customer invoice that **regroups delivery notes already counted** (`RelatedDocuments`);
  counting both would double the demand.
- `flag_qty_nonpositive`, `flag_price_nonpositive`: kept visible for the anomaly team, not deleted here.

Lines with no document or comment lines go to `quarantine`; drafts and grouped invoices are excluded from demand.

**What to tell the professor:** "We separated structural errors (quarantine) from business exclusions (drafts,
double counting) and kept every decision countable."

In [6]:
# STEP 2B: Keep customer sales and returns, flag every exclusion.
SALE_TYPES, RETURN_TYPES = {'1', '5', '6'}, {'4', '7'}
linked = load_csv(RAW_DIR / 'RelatedDocuments.csv')
invoice_ids = set(docs.loc[docs['DocumentType_Id'] == '1', 'Id'])
grouped_invoices = set(linked['Document1_Id']) & invoice_ids

lines = items.merge(docs[['Id', 'DocumentDate', 'DocumentType_Id', 'State', 'origin']],
                    how='left', left_on='Document_Id', right_on='Id', suffixes=('', '_doc'),
                    validate='many_to_one')
assert len(lines) == len(items)
lines['flag_no_document'] = lines['Id_doc'].isna()
lines['flag_comment_line'] = lines['ProductStock_Id'].isna() & lines['PrixTotalHt'].fillna(0).eq(0)
lines['flag_draft'] = lines['State'].ne('1')
lines['flag_grouped_invoice'] = lines['Document_Id'].isin(grouped_invoices)
lines['flag_qty_nonpositive'] = lines['Quantity'].le(0)
lines['flag_price_nonpositive'] = lines['PrixHt'].le(0)
lines['is_customer_flow'] = lines['DocumentType_Id'].isin(SALE_TYPES | RETURN_TYPES)

flags = [c for c in lines if c.startswith('flag_')]
display(lines.loc[lines['is_customer_flow'], flags].sum().to_frame('flagged_customer_lines'))
quarantine = lines[lines['flag_no_document'] | lines['flag_comment_line']]
quarantine.to_csv(OUTPUT_DIR / 'lines_quarantine.csv', index=False)
sales = lines[lines['is_customer_flow'] & ~lines['flag_no_document'] & ~lines['flag_comment_line']
              & ~lines['flag_draft'] & ~lines['flag_grouped_invoice']].copy()
print(f'Customer lines kept: {len(sales):,} | quarantined lines: {len(quarantine):,}')

,flagged_customer_lines
flag_no_document,0
flag_comment_line,1330
flag_draft,1026
flag_grouped_invoice,5638
flag_qty_nonpositive,438
flag_price_nonpositive,2


Customer lines kept: 91,057 | quarantined lines: 1,381


## 3 — Controlled Data Transformation

1. **Line → product:** `ProductStock_Id → ProductStocks.Id → Product_Id` (one stock line per product: one store).
2. **Product → family / brand:** from the cleaned catalogue, after checking that `Product_Id` is unique.
3. **Signed quantity:** returns are negative, so a week's demand is net of returns.
4. **Weekly aggregation** on a **complete calendar**: a week without sales is a real **0**, not a missing value.

### Guided code explanation — Joins, signed quantities and the weekly grid

- `safe_left_merge()` refuses a lookup table with duplicate keys and asserts the row count did not change.
- The catalogue has a few products sharing a `Product_Id`: we keep one row per id and report how many were dropped.
- `week_start` = the Monday of the document date (`date − weekday`).
- `LEVEL` chooses the forecast level: `'FamilyGroup'` (7 families) or `'Marque'` (26 best-selling brands, the rest
  grouped into `AUTRES MARQUES`).
- The grid is built with `pd.MultiIndex.from_product(weeks × groups)`, then `fillna(0)` for quantities: **this is the
  missing-value treatment of the target series**.
- `week_origin` is `real` only when the whole week is ≤ 30/04/2025; the week 28/04 → 04/05 mixes both periods.
- Lines whose product has no match in the catalogue (about 7 %) are kept in a group `NON RENSEIGNE` instead of being
  dropped: removing them would under-estimate total demand.

**Expected result:** one row per week and group, with no duplicate, and the number of real weeks.

In [7]:
# STEP 3: Line -> product -> family / brand, signed quantities, complete weekly grid.
LEVEL = 'FamilyGroup'          # or 'Marque'

stock_map = raw['product_stocks'][['Id', 'Product_Id']].rename(columns={'Id': 'stock_id', 'Product_Id': 'Product_Id_ps'})
sales = safe_left_merge(sales, stock_map, 'ProductStock_Id', 'stock_id', 'stock line -> product')

cat = catalogue.dropna(subset=['Product_Id'])
print('Catalogue rows sharing a Product_Id (kept once):', int(cat['Product_Id'].duplicated().sum()))
cat = cat.drop_duplicates('Product_Id')[['Product_Id', 'FamilyGroup', 'Marque']]
cat = cat.rename(columns={'Product_Id': 'cat_product_id'})
sales = safe_left_merge(sales, cat, 'Product_Id_ps', 'cat_product_id', 'product -> family/brand')
sales['FamilyGroup'] = sales['FamilyGroup'].fillna('NON RENSEIGNE')
sales['Marque'] = sales['Marque'].fillna('NON RENSEIGNE')

sign = np.where(sales['DocumentType_Id'].isin(RETURN_TYPES), -1, 1)
sales['qty'] = sign * sales['Quantity']
sales['revenue_ht'] = sign * sales['PrixTotalHt']
sales['discount_ht'] = sign * sales['SumDiscountTotal'].fillna(0)
sales['is_counter'] = sales['DocumentType_Id'].eq('6')
sales['week_start'] = sales['DocumentDate'].dt.normalize() - pd.to_timedelta(sales['DocumentDate'].dt.weekday, unit='D')

if LEVEL == 'Marque':
    top = sales.groupby('Marque')['qty'].sum().drop('NON RENSEIGNE', errors='ignore').nlargest(26).index
    sales['group'] = sales['Marque'].where(sales['Marque'].isin(top), 'AUTRES MARQUES')
else:
    sales['group'] = sales['FamilyGroup']

weekly = (sales.groupby(['group', 'week_start'])
          .agg(qty=('qty', 'sum'), revenue_ht=('revenue_ht', 'sum'), discount_ht=('discount_ht', 'sum'),
               counter_qty=('qty', lambda q: q[sales.loc[q.index, 'is_counter']].sum()),
               n_documents=('Document_Id', 'nunique'))
          .reset_index())
end = SYNTH_END if INCLUDE_SYNTHETIC else REAL_END
weeks = pd.date_range(REAL_START, end, freq='W-MON')
grid = pd.MultiIndex.from_product([sorted(sales['group'].unique()), weeks], names=['group', 'week_start'])
weekly = weekly.set_index(['group', 'week_start']).reindex(grid).reset_index()
filled = int(weekly['qty'].isna().sum())
weekly[['qty', 'revenue_ht', 'discount_ht', 'counter_qty', 'n_documents']] = (
    weekly[['qty', 'revenue_ht', 'discount_ht', 'counter_qty', 'n_documents']].fillna(0))
weekly['week_complete'] = (weekly['week_start'] + pd.Timedelta(days=6)) <= end
weekly['week_origin'] = np.where(weekly['week_start'] + pd.Timedelta(days=6) <= REAL_END, 'real', 'synthetic')
assert not weekly.duplicated(['group', 'week_start']).any()
print(f'{len(weekly):,} rows = {weekly.group.nunique()} groups × {len(weeks)} weeks | weeks without sales filled with 0: {filled}')
display(weekly.groupby('week_origin')['week_start'].agg(['min', 'max', 'nunique']))

stock line -> product         91,057 rows kept | 100.0 % matched
Catalogue rows sharing a Product_Id (kept once): 4


product -> family/brand       91,057 rows kept |  92.5 % matched


735 rows = 7 groups × 105 weeks | weeks without sales filled with 0: 197


,min,max,nunique
week_origin,,,
real,2024-09-30,2025-04-21,30
synthetic,2025-04-28,2026-09-28,75


## 4 — Feature Engineering

| Family | Features | Known at the end of week t? |
|---|---|---|
| Target | `y_next_qty` = quantity of week t+1 | — (what we predict) |
| Past demand | `qty`, `qty_lag1`, `qty_lag3`, `qty_ma4`, `qty_trend` | yes |
| Sales context | `avg_price_ht`, `discount_rate_pct`, `counter_share_pct`, `n_documents` | yes |
| Calendar of t+1 | `next_holidays`, `next_working_days`, `next_month`, `next_week_sin/cos` | yes (calendar is fixed) |
| Weather of t | `w_temp_mean`, `w_temp_min`, `w_temp_max`, `w_rain_mm`, `w_rainy_days`, `w_sunshine_h`, `w_radiation_mj` | yes (observed) |
| Epidemics of t−1 | `e_flu`, `e_resp_infection`, `e_diarrhoea`, `e_chickenpox` | yes (published with ~1 week delay) |

### Guided code explanation — Feature engineering

- `groupby('group').shift(k)` takes the value **k weeks earlier inside the same family**: lags never mix families.
- `qty_ma4` is the mean of the current and three previous weeks; `qty_trend = qty − qty_lag1`.
- `y_next_qty = shift(-1)`: the quantity of the **next** week. The last week of each series has no target and is
  dropped from training.
- French public holidays are **computed** (fixed dates + Easter Monday, Ascension, Whit Monday); school holidays are
  deliberately not used (project rule).
- `np.sin / np.cos(2π·week/52)` encode the ISO week as a circle, so week 52 is close to week 1.
- Weather: daily values → weekly mean (temperature), min, max, total (rain, sunshine, radiation) and rainy days (≥ 1 mm).
- Epidemics: `inc100` per ISO week, then **shifted by one week** to respect the publication delay.

**Leakage rule:** no feature uses information from week t+1 except the calendar, which is known in advance.

In [8]:
# STEP 4: Build the features. Everything is computed inside each group, in chronological order.
f = weekly.sort_values(['group', 'week_start']).reset_index(drop=True)
g = f.groupby('group')['qty']
f['qty_lag1'] = g.shift(1)
f['qty_lag3'] = g.shift(3)
f['qty_ma4'] = g.transform(lambda s: s.rolling(4, min_periods=4).mean())
f['qty_trend'] = f['qty'] - f['qty_lag1']
f['avg_price_ht'] = np.where(f['qty'] > 0, f['revenue_ht'] / f['qty'].where(f['qty'] > 0), np.nan)
gross = f['revenue_ht'] + f['discount_ht']
f['discount_rate_pct'] = np.where(gross > 0, 100 * f['discount_ht'] / gross.where(gross > 0), 0)
f['counter_share_pct'] = np.where(f['qty'] > 0, 100 * f['counter_qty'] / f['qty'].where(f['qty'] > 0), np.nan)
f['y_next_qty'] = f.groupby('group')['qty'].shift(-1)

# French public holidays (computed, no external file).
def easter(y):
    a, b, c = y % 19, y // 100, y % 100
    d, e = b // 4, b % 4
    g_, h = (8 * b + 13) // 25, (19 * a + b - (b // 4) - (8 * b + 13) // 25 + 15) % 30
    i, k = c // 4, c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month, day = divmod(h + l - 7 * m + 114, 31)
    return pd.Timestamp(y, month, day + 1)

def holidays(y):
    e = easter(y)
    fixed = [(1, 1), (5, 1), (5, 8), (7, 14), (8, 15), (11, 1), (11, 11), (12, 25)]
    return {pd.Timestamp(y, m, d) for m, d in fixed} | {e + pd.Timedelta(days=1), e + pd.Timedelta(days=39),
                                                        e + pd.Timedelta(days=50)}

days = pd.DataFrame({'date': pd.date_range(REAL_START, SYNTH_END + pd.Timedelta(days=7))})
hol = set().union(*(holidays(y) for y in days.date.dt.year.unique()))
days['holiday'] = days['date'].isin(hol)
days['working'] = (days['date'].dt.weekday < 6) & ~days['holiday']      # the store opens Monday-Saturday
days['week_start'] = days['date'] - pd.to_timedelta(days['date'].dt.weekday, unit='D')
cal = days.groupby('week_start').agg(holidays=('holiday', 'sum'), working_days=('working', 'sum')).reset_index()
cal['week_start'] = cal['week_start'] - pd.Timedelta(days=7)            # calendar of week t+1, attached to week t
cal = cal.rename(columns={'holidays': 'next_holidays', 'working_days': 'next_working_days'})
f = f.merge(cal, on='week_start', how='left', validate='many_to_one')
next_week = f['week_start'] + pd.Timedelta(days=7)
f['next_month'] = next_week.dt.month
iso = next_week.dt.isocalendar().week.astype(int)
f['next_week_sin'] = np.sin(2 * np.pi * iso / 52)
f['next_week_cos'] = np.cos(2 * np.pi * iso / 52)

# Weather of week t: daily -> weekly.
w = weather.copy()
w['week_start'] = w['date'] - pd.to_timedelta(w['date'].dt.weekday, unit='D')
w_week = w.groupby('week_start').agg(
    w_temp_mean=('temp_mean', 'mean'), w_temp_min=('temp_min', 'min'), w_temp_max=('temp_max', 'max'),
    w_rain_mm=('rain_mm', 'sum'), w_rainy_days=('rain_mm', lambda s: int((s >= 1).sum())),
    w_sunshine_h=('sunshine_s', lambda s: s.sum() / 3600), w_radiation_mj=('radiation_mj', 'sum')).reset_index()
f = f.merge(w_week, on='week_start', how='left', validate='many_to_one')

# Epidemics of week t-1 (publication delay).
epi = epidemics_raw.copy()
epi['inc100'] = pd.to_numeric(epi['inc100'], errors='coerce')
epi['week_start'] = pd.to_datetime(epi['week'].str[:4] + '-W' + epi['week'].str[4:] + '-1', format='%G-W%V-%u')
epi = epi.pivot_table(index='week_start', columns='indicator', values='inc100').add_prefix('e_').reset_index()
epi['week_start'] = epi['week_start'] + pd.Timedelta(days=7)            # value of week t-1 seen at week t
f = f.merge(epi, on='week_start', how='left', validate='many_to_one')

display(f[['group', 'week_start', 'qty', 'qty_lag1', 'qty_ma4', 'next_holidays', 'w_temp_mean', 'e_flu',
           'y_next_qty']].head(8))

,group,week_start,qty,qty_lag1,qty_ma4,next_holidays,w_temp_mean,e_flu,y_next_qty
0,ACCESSOIRE,2024-09-30,0.0,NaN,NaN,0,12.385714,137.0,0.0
1,ACCESSOIRE,2024-10-07,0.0,0.0,NaN,0,12.971429,127.0,0.0
2,ACCESSOIRE,2024-10-14,0.0,0.0,NaN,0,15.442857,119.0,0.0
3,ACCESSOIRE,2024-10-21,0.0,0.0,0.00,1,13.400000,102.0,0.0
4,ACCESSOIRE,2024-10-28,0.0,0.0,0.00,0,11.500000,70.0,1.0
5,ACCESSOIRE,2024-11-04,1.0,0.0,0.25,1,10.042857,54.0,3.0
6,ACCESSOIRE,2024-11-11,3.0,1.0,1.00,0,8.171429,71.0,2.0
7,ACCESSOIRE,2024-11-18,2.0,3.0,1.50,0,6.485714,85.0,1.0


## 5 — Feature Selection and Leakage Audit

Selection uses **real weeks only**, because the generated months were produced with assumptions about weather
and epidemics; measuring their effect on synthetic weeks would only rediscover our own assumptions.

### Guided code explanation — Feature selection

- `train_rows` = complete weeks with a known target; `real_rows` = the subset whose week is real.
- `relevance` = Spearman correlation of each candidate with `y_next_qty`, **computed inside each family** and
  averaged (otherwise the size of a family would dominate the correlation).
- `redundancy`: pairs of features with |r| > 0.9 carry the same information. The weather block is expected to be
  highly correlated (temperature mean/min/max, sunshine, radiation) — this motivates the PCA of Step 6.
- `constant` features are removed; identifiers (`group`, dates) are kept only to trace rows.
- The `leakage_audit` table lists the variables that **must not** be used: `y_next_qty` itself, any weather or
  epidemic value of week t+1.

**What to say:** "External variables are candidates: they are kept only if they improve the forecast on real weeks."

In [9]:
# STEP 5: Relevance (real weeks), redundancy, constants, leakage audit.
PAST = ['qty', 'qty_lag1', 'qty_lag3', 'qty_ma4', 'qty_trend']
CONTEXT = ['avg_price_ht', 'discount_rate_pct', 'counter_share_pct', 'n_documents']
CALENDAR = ['next_holidays', 'next_working_days', 'next_month', 'next_week_sin', 'next_week_cos']
WEATHER = ['w_temp_mean', 'w_temp_min', 'w_temp_max', 'w_rain_mm', 'w_rainy_days', 'w_sunshine_h', 'w_radiation_mj']
EPIDEMICS = ['e_flu', 'e_resp_infection', 'e_diarrhoea', 'e_chickenpox']
CANDIDATES = PAST + CONTEXT + CALENDAR + WEATHER + EPIDEMICS

train_rows = f[f['week_complete'] & f['y_next_qty'].notna() & f['qty_lag3'].notna()].copy()
next_is_real = (train_rows['week_start'] + pd.Timedelta(days=13)) <= REAL_END
real_rows = train_rows[next_is_real]
print(f'Model rows: {len(train_rows):,} | rows whose week AND target week are real: {len(real_rows):,}')

per_group = [real_rows[real_rows.group == grp][CANDIDATES + ['y_next_qty']].corr(method='spearman')['y_next_qty']
             for grp in real_rows.group.unique()]
relevance = pd.concat(per_group, axis=1).drop('y_next_qty').mean(axis=1).rename('mean_spearman_r_real_weeks')
missing = train_rows[CANDIDATES].isna().mean().mul(100).rename('missing_%')
selection = pd.concat([relevance, missing], axis=1)
selection['block'] = ['past'] * len(PAST) + ['context'] * len(CONTEXT) + ['calendar'] * len(CALENDAR) \
                     + ['weather'] * len(WEATHER) + ['epidemics'] * len(EPIDEMICS)
display(selection.round(3).sort_values('mean_spearman_r_real_weeks', key=abs, ascending=False))

corr = train_rows[CANDIDATES].corr()
pairs = [(a, b, round(corr.loc[a, b], 3)) for i, a in enumerate(CANDIDATES) for b in CANDIDATES[i + 1:]
         if abs(corr.loc[a, b]) > 0.9]
display(pd.DataFrame(pairs, columns=['feature_a', 'feature_b', 'r']))
constant = [c for c in CANDIDATES if train_rows[c].nunique(dropna=True) <= 1]
print('Constant features removed:', constant)

leakage_audit = pd.DataFrame([
    ['y_next_qty', 'target', 'never an input'],
    ['weather of week t+1', 'future observation', 'not computed'],
    ['epidemics of week t', 'published one week later', 'replaced by week t-1'],
    ['group, week_start', 'identifiers', 'traceability only (group is one-hot encoded)'],
], columns=['variable', 'reason', 'treatment'])
display(leakage_audit)
selection.to_csv(OUTPUT_DIR / 'feature_selection.csv', index_label='feature')

Model rows: 707 | rows whose week AND target week are real: 182


,mean_spearman_r_real_weeks,missing_%,block
w_rain_mm,0.206,0.00,weather
w_rainy_days,0.184,0.00,weather
n_documents,0.149,0.00,context
w_temp_max,0.137,0.00,weather
avg_price_ht,0.126,26.45,context
next_week_sin,-0.121,0.00,calendar
e_diarrhoea,-0.116,0.00,epidemics
counter_share_pct,0.104,26.45,context
next_working_days,-0.083,0.00,calendar
next_holidays,0.083,0.00,calendar


,feature_a,feature_b,r
0,qty,qty_lag1,0.964
1,qty,qty_lag3,0.956
2,qty,qty_ma4,0.984
3,qty,n_documents,0.988
4,qty_lag1,qty_lag3,0.957
5,qty_lag1,qty_ma4,0.986
6,qty_lag1,n_documents,0.968
7,qty_lag3,qty_ma4,0.984
8,qty_lag3,n_documents,0.959
9,qty_ma4,n_documents,0.983


Constant features removed: []


,variable,reason,treatment
0,y_next_qty,target,never an input
1,weather of week t+1,future observation,not computed
2,epidemics of week t,published one week later,replaced by week t-1
3,"group, week_start",identifiers,traceability only (group is one-hot encoded)


## 6 — Imputation, One-Hot Encoding, Standardization and PCA

1. **Missing numeric values → training median.** They exist for `avg_price_ht` and `counter_share_pct` when a
   family sold nothing that week (a price is undefined when quantity is 0).
2. **Family / brand → one-hot encoding.**
3. **StandardScaler** on numeric features.
4. **Dimensionality reduction — PCA of the weather block:** seven correlated weather variables are replaced by the
   few components that keep at least 90 % of their variance.
5. Every transformer is fitted on the **earlier** weeks only, then applied unchanged to later weeks.

### Guided code explanation — Preprocessing and PCA

- **Chronological split:** the first 80 % of weeks train, the last 20 % are a holdout. Never shuffle weeks: it would
  let the model see the future.
- `weather_pca = Pipeline([impute, scale, PCA(0.90)])` compresses the weather block. The **loadings** table shows what
  each component means (for example PC1 ≈ "warm and sunny weeks" vs "cold and dark weeks").
- `ColumnTransformer` applies three treatments side by side: numeric pipeline, weather PCA, one-hot of the group.
- `get_feature_names_out()` keeps the name of every final column so each model input stays explainable.

**Expected output:** the number of weather components, their explained variance, and the final matrix shape.

**Limitation:** PCA keeps variance, not predictive power: Step 7 compares the forecast with and without it.

In [10]:
# STEP 6: Chronological split, then fit every transformer on the training weeks only.
ordered = train_rows.sort_values('week_start').reset_index(drop=True)
cut_week = ordered['week_start'].quantile(0.8)
train = ordered[ordered['week_start'] <= cut_week].copy()
holdout = ordered[ordered['week_start'] > cut_week].copy()
print(f'Train weeks: {train.week_start.min():%d/%m/%Y} → {train.week_start.max():%d/%m/%Y} ({len(train)} rows) | '
      f'holdout: {holdout.week_start.min():%d/%m/%Y} → {holdout.week_start.max():%d/%m/%Y} ({len(holdout)} rows)')

NUMERIC = [c for c in PAST + CONTEXT + CALENDAR + EPIDEMICS if c not in constant]
num_pipe = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
weather_pca = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                        ('pca', PCA(n_components=0.90, svd_solver='full'))])
preprocessor = ColumnTransformer([
    ('num', num_pipe, NUMERIC),
    ('weather', weather_pca, WEATHER),
    ('group', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['group']),
], remainder='drop', sparse_threshold=0)
X_train = preprocessor.fit_transform(train)
X_holdout = preprocessor.transform(holdout)
names = preprocessor.get_feature_names_out()
assert np.isfinite(X_train).all() and np.isfinite(X_holdout).all()

pca = preprocessor.named_transformers_['weather'].named_steps['pca']
print(f'Weather block: {len(WEATHER)} variables → {pca.n_components_} components, '
      f'explained variance {pca.explained_variance_ratio_.sum():.3f}')
loadings = pd.DataFrame(pca.components_.T, index=WEATHER,
                        columns=[f'PC{i + 1}' for i in range(pca.n_components_)]).round(2)
display(loadings)
print('Final matrix:', X_train.shape, '| holdout:', X_holdout.shape)
print('Columns:', list(names))

Train weeks: 21/10/2024 → 04/05/2026 (567 rows) | holdout: 11/05/2026 → 21/09/2026 (140 rows)


Weather block: 7 variables → 3 components, explained variance 0.951


,PC1,PC2,PC3
w_temp_mean,0.45,0.14,0.29
w_temp_min,0.42,0.16,0.59
w_temp_max,0.45,0.13,0.08
w_rain_mm,-0.10,0.69,-0.22
w_rainy_days,-0.14,0.68,-0.03
w_sunshine_h,0.43,-0.05,-0.54
w_radiation_mj,0.45,0.00,-0.46


Final matrix: (567, 28) | holdout: (140, 28)
Columns: ['num__qty', 'num__qty_lag1', 'num__qty_lag3', 'num__qty_ma4', 'num__qty_trend', 'num__avg_price_ht', 'num__discount_rate_pct', 'num__counter_share_pct', 'num__n_documents', 'num__next_holidays', 'num__next_working_days', 'num__next_month', 'num__next_week_sin', 'num__next_week_cos', 'num__e_flu', 'num__e_resp_infection', 'num__e_diarrhoea', 'num__e_chickenpox', 'weather__pca0', 'weather__pca1', 'weather__pca2', 'group__group_ACCESSOIRE', 'group__group_ANIMALERIE', 'group__group_AUTRES', 'group__group_COMPLEMENTS ALIMENTAIRES', 'group__group_NON CLASSE', 'group__group_NON RENSEIGNE', 'group__group_TOILETTE']


## 7 — Visual Evidence for the Presentation

Charts generated from the files: weekly demand per group (real vs synthetic period), correlation of the weather
block, and the variance kept by the weather PCA.

In [11]:
# STEP 7: Plots saved in outputs/ for the slides.
fig, ax = plt.subplots(figsize=(11, 4.5))
for grp, d in f.groupby('group'):
    ax.plot(d['week_start'], d['qty'], label=grp, linewidth=1.2)
ax.axvspan(REAL_START, REAL_END, color='grey', alpha=0.12, label='real period')
ax.set(title=f'Weekly quantity sold per {LEVEL}', xlabel='week', ylabel='net quantity')
ax.legend(ncol=4, fontsize=8)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'weekly_demand.png', dpi=160); plt.show()

wc = train_rows[WEATHER + EPIDEMICS].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(wc, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(wc)), wc.columns, rotation=90); ax.set_yticks(range(len(wc)), wc.columns)
fig.colorbar(im, ax=ax); ax.set_title('External variables: correlation')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'external_correlation.png', dpi=160); plt.show()

fig, ax = plt.subplots(figsize=(6, 3.5))
full = PCA().fit(StandardScaler().fit_transform(train[WEATHER].fillna(train[WEATHER].median())))
ax.plot(range(1, len(WEATHER) + 1), np.cumsum(full.explained_variance_ratio_), marker='o')
ax.axhline(0.9, linestyle='--', label='90 % threshold')
ax.set(xlabel='components', ylabel='cumulative explained variance', title='Weather PCA')
ax.legend(); plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'weather_pca_variance.png', dpi=160); plt.show()

C:\Users\firas\AppData\Local\Temp\ipykernel_41856\2371330663.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'weekly_demand.png', dpi=160); plt.show()


C:\Users\firas\AppData\Local\Temp\ipykernel_41856\2371330663.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'external_correlation.png', dpi=160); plt.show()


C:\Users\firas\AppData\Local\Temp\ipykernel_41856\2371330663.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'weather_pca_variance.png', dpi=160); plt.show()


## 8 — Final Validation and Deliverables

- One row per week × group, no duplicates, complete calendar.
- Drafts and regrouped invoices excluded; returns subtracted; quarantined lines exported.
- Every conversion audited; no target information in the inputs.
- Imputer, scaler, encoder and PCA fitted on training weeks only.
- Real and synthetic weeks tagged; external variables judged on real weeks.

In [12]:
# STEP 8: Integrity checks and exports.
assert not f.duplicated(['group', 'week_start']).any()
assert f.groupby('group').size().nunique() == 1, 'every group must have the same number of weeks'
assert 'y_next_qty' not in names
f.to_csv(OUTPUT_DIR / 'forecast_feature_table.csv', index=False)
pd.DataFrame(X_train, columns=names).to_csv(OUTPUT_DIR / 'X_train_prepared.csv', index=False)
pd.DataFrame(X_holdout, columns=names).to_csv(OUTPUT_DIR / 'X_holdout_prepared.csv', index=False)
summary = pd.DataFrame([
    ['customer_lines_kept', len(sales)], ['quarantined_lines', len(quarantine)],
    ['groups', f.group.nunique()], ['weeks', f.week_start.nunique()],
    ['real_weeks', int((f.week_origin == 'real').sum() / f.group.nunique())],
    ['model_rows', len(train_rows)], ['prepared_features', X_train.shape[1]],
    ['weather_components', int(pca.n_components_)],
], columns=['metric', 'value'])
display(summary)
summary.to_csv(OUTPUT_DIR / 'preparation_summary.csv', index=False)
print('Saved all outputs to:', OUTPUT_DIR.resolve())

,metric,value
0,customer_lines_kept,91057
1,quarantined_lines,1381
2,groups,7
3,weeks,105
4,real_weeks,30
5,model_rows,707
6,prepared_features,28
7,weather_components,3


Saved all outputs to: C:\Users\firas\OneDrive\Documents\ssis\outputs\repository-migration\payloads\ml-notebooks\outputs\demand_forecast


# 9 — Optional extension: does the forecast beat "same as last week"?

**Rule of the project (cahier §8):** external variables are kept only if they improve the forecast **on the real
period**. We therefore run the comparison on real weeks only: the first 70 % of real weeks train, the last 30 % test.

Three candidates:
1. **Baseline:** next week = this week (`qty`).
2. **Ridge regression** on past demand, context and calendar.
3. **Ridge regression + weather PCA + epidemics.**

Metrics: **MAE** (average error in units) and **RMSE** (penalises large misses). With ~25 real weeks per family the
test set is small: results are indicative, not a final evaluation.

In [13]:
# STEP 9: Baseline vs models, evaluated on real weeks only.
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

r = real_rows.sort_values('week_start')
cut = r['week_start'].quantile(0.7)
r_train, r_test = r[r.week_start <= cut], r[r.week_start > cut]

def make(features, weather=False):
    parts = [('num', num_pipe, features), ('group', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['group'])]
    if weather:
        parts.append(('weather', weather_pca, WEATHER))
    return Pipeline([('prep', ColumnTransformer(parts, sparse_threshold=0)), ('model', Ridge(alpha=1.0))])

results, preds = [], {}
def score(name, y_pred):
    preds[name] = y_pred
    results.append([name, mean_absolute_error(r_test.y_next_qty, y_pred),
                    mean_squared_error(r_test.y_next_qty, y_pred) ** 0.5])

score('baseline: same as this week', r_test['qty'])
internal = [c for c in PAST + CONTEXT + CALENDAR if c not in constant]
score('ridge: internal + calendar', make(internal).fit(r_train, r_train.y_next_qty).predict(r_test))
score('ridge: + weather PCA + epidemics',
      make(internal + EPIDEMICS, weather=True).fit(r_train, r_train.y_next_qty).predict(r_test))
results = pd.DataFrame(results, columns=['model', 'MAE', 'RMSE']).round(2)
display(results)
results.to_csv(OUTPUT_DIR / 'forecast_comparison_real_weeks.csv', index=False)
print(f'Train: {r_train.week_start.nunique()} real weeks | test: {r_test.week_start.nunique()} real weeks')

,model,MAE,RMSE
0,baseline: same as this week,45.73,94.07
1,ridge: internal + calendar,61.71,98.98
2,ridge: + weather PCA + epidemics,63.58,101.03


Train: 19 real weeks | test: 7 real weeks


### Reading the result

Compare the three MAE values of the table above:
- If the **baseline** has the lowest error, the models have not learned more than "next week looks like this week".
  With only ~19 real training weeks per family this is a normal outcome, and it is a result worth presenting: it sets
  the bar any future model must beat.
- If the line **with weather PCA + epidemics** is not better than the internal model, the §8 rule says the external
  variables are **not retained** for now. They stay in the prepared dataset so the test can be repeated when more
  real weeks are available.

Do not compare these numbers with errors measured on synthetic weeks: the generated months follow our own
assumptions and would flatter the models.

In [14]:
# STEP 9B: Forecast vs actual on the test weeks, for the largest group.
big = r_test.groupby('group')['qty'].sum().idxmax()
mask = (r_test['group'] == big).to_numpy()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(r_test.week_start[mask], r_test.y_next_qty[mask], marker='o', label='actual (next week)')
for name, p in preds.items():
    ax.plot(r_test.week_start[mask], np.asarray(p)[mask], marker='.', linestyle='--', label=name)
ax.set(title=f'Next-week forecast — {big} (real test weeks)', xlabel='week t', ylabel='quantity in week t+1')
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'forecast_vs_actual.png', dpi=160); plt.show()

C:\Users\firas\AppData\Local\Temp\ipykernel_41856\2807047084.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'forecast_vs_actual.png', dpi=160); plt.show()


## 10 — What to say during the presentation

**Person 1:** "We kept validated customer sales only, subtracted returns, removed invoices that regroup delivery
notes already counted, and built one row per week and family on a complete calendar, where a week without sales is a
real zero. We then created lags and a moving average computed strictly on past weeks."

**Person 2:** "We added the calendar of the coming week, the observed weather of the current week and the epidemic
indicators of the previous week, to respect what is really known when the order is placed. We imputed with training
medians, one-hot encoded the families, scaled the numeric inputs and compressed the seven correlated weather
variables with a PCA."

**Both:** "External variables are candidates. They stay only if they beat the baseline on real weeks."

### Limitations to disclose
- Only ~30 real weeks: one seasonal cycle is incomplete in the real data; the generated months help build the
  pipeline but cannot validate it.
- Weather is for Paris; the store's exact location may differ.
- The test set of Step 9 is small; MAE/RMSE are indicative.
- Ridge is a simple, explainable candidate; tree-based or time-series models can be compared later with the same
  prepared data.